# 03 - Wearables & Photoplethysmography (PPG) Analysis
## MedAgent: Multimodal Clinical Decision Support System

This notebook demonstrates pulsatile PPG signal processing, systolic peak detection, Pulse Rate Variability (PRV), and continuous physiological monitoring for wearable integration.


In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import find_peaks


### 1. Synthetic Pulsatile PPG Signal Model
Simulates systolic peak, dicrotic notch, and diastolic peak.


In [ ]:
fs = 100  # 100 Hz wearable sampling frequency
duration = 10
t = np.arange(0, duration, 1/fs)
hr_bpm = 72
rr_interval = 60.0 / hr_bpm

# Pulsatile waveform synthesis
ppg_clean = np.zeros_like(t)
for beat_time in np.arange(0, duration, rr_interval):
    idx = (t >= beat_time) & (t < beat_time + rr_interval)
    t_beat = t[idx] - beat_time
    # Primary systolic wave + secondary dicrotic reflection
    systolic = 1.2 * np.exp(-((t_beat - 0.15) ** 2) / 0.008)
    dicrotic = 0.45 * np.exp(-((t_beat - 0.38) ** 2) / 0.012)
    ppg_clean[idx] = systolic + dicrotic

# Add physiological baseline respiratory modulation
respiration = 0.15 * np.sin(2 * np.pi * 0.25 * t)
noise = np.random.normal(0, 0.02, len(t))
ppg_signal = ppg_clean + respiration + noise

# Detect systolic peaks
peaks, _ = find_peaks(ppg_signal, distance=int(fs * 0.5), prominence=0.4)

plt.figure(figsize=(15, 5))
plt.plot(t, ppg_signal, label="PPG Pulsatile Waveform (100 Hz)", color="#e11d48", lw=1.5)
plt.scatter(t[peaks], ppg_signal[peaks], color="#1e3a8a", s=60, zorder=5, label="Systolic Peaks")
plt.title("Wearable Photoplethysmography (PPG) & Peak Detection", fontsize=14)
plt.xlabel("Time (seconds)", fontsize=12)
plt.ylabel("Optical Density (A.U.)", fontsize=12)
plt.grid(True, alpha=0.3)
plt.legend(loc="upper right")
plt.tight_layout()
plt.show()


### 2. Pulse Rate Variability (PRV) Analysis


In [ ]:
peak_times = t[peaks]
ibi = np.diff(peak_times) * 1000  # Inter-beat intervals in ms
instant_hr = 60000.0 / ibi

print(f"Mean Inter-Beat Interval (IBI): {np.mean(ibi):.1f} ms")
print(f"Calculated Heart Rate: {np.mean(instant_hr):.1f} BPM (Std: {np.std(instant_hr):.2f})")
print(f"RMSSD (Parasympathetic index): {np.sqrt(np.mean(np.square(np.diff(ibi)))):.2f} ms")
